In [1]:
# ============================================================
# LIVE DEMO 1 — PYTHON TOOL CALLING
# ============================================================
#
# Goal:
# Demonstrate the architecture:
#
# REASON
#   ↓
# TOOL
#   ↓
# OBSERVATION
#   ↓
# UPDATE STATE
#   ↓
# NEXT DECISION
#
# We deliberately keep the tools simple so students focus
# on agent architecture rather than Python complexity.
# ============================================================


# ------------------------------------------------------------
# TOOL 1: GET SERVER HEALTH
# ------------------------------------------------------------

def get_server_health(server: str):
    """
    Simulates retrieving live health information
    for an authorized server.

    Parameters
    ----------
    server : str
        Server name.

    Returns
    -------
    dict
        Server health metrics.
    """

    print("\n[TOOL CALLED] get_server_health")
    print(f"Server: {server}")

    # Simulated monitoring database
    server_data = {

        "APP-PROD-12": {
            "cpu": 96,
            "memory": 71,
            "db_connections": 100,
            "status": "DEGRADED"
        },

        "APP-PROD-11": {
            "cpu": 42,
            "memory": 65,
            "db_connections": 54,
            "status": "HEALTHY"
        },

        "WEB-PROD-01": {
            "cpu": 55,
            "memory": 61,
            "db_connections": 38,
            "status": "HEALTHY"
        }
    }

    # Check whether the server exists
    if server not in server_data:
        return {
            "error": f"Server '{server}' not found."
        }

    return server_data[server]


# ------------------------------------------------------------
# TOOL 2: SEARCH INCIDENTS
# ------------------------------------------------------------

def search_incidents(query: str):
    """
    Simulates searching historical incidents.

    Parameters
    ----------
    query : str
        Natural-language search query.

    Returns
    -------
    list
        Matching incident records.
    """

    print("\n[TOOL CALLED] search_incidents")
    print(f"Query: {query}")

    # Simulated historical incident database
    incidents = [

        {
            "ticket_id": "INC-6731",
            "server": "APP-PROD-12",
            "title": "Connection pool exhaustion",
            "symptoms": [
                "database connections high",
                "application slow",
                "requests timing out"
            ],
            "resolution": "Restart application connection pool"
        },

        {
            "ticket_id": "INC-6625",
            "server": "APP-PROD-12",
            "title": "High CPU utilization",
            "symptoms": [
                "cpu high",
                "application latency"
            ],
            "resolution": "Restart worker process and investigate runaway thread"
        },

        {
            "ticket_id": "INC-6519",
            "server": "WEB-PROD-01",
            "title": "Memory pressure",
            "symptoms": [
                "memory high",
                "server slow"
            ],
            "resolution": "Restart application and investigate memory leak"
        }
    ]

    # Convert query to lowercase to simplify matching
    query_lower = query.lower()

    results = []

    # Very simple keyword-based search
    for incident in incidents:

        searchable_text = (
            incident["server"] + " "
            + incident["title"] + " "
            + " ".join(incident["symptoms"])
        ).lower()

        # Count how many important words overlap
        query_words = query_lower.split()

        score = 0

        for word in query_words:
            if word in searchable_text:
                score += 1

        if score > 0:
            incident_copy = incident.copy()
            incident_copy["match_score"] = score
            results.append(incident_copy)

    # Sort strongest matches first
    results.sort(
        key=lambda x: x["match_score"],
        reverse=True
    )

    return results


# ------------------------------------------------------------
# TOOL 3: GET INCIDENT
# ------------------------------------------------------------

def get_incident(ticket_id: str):
    """
    Retrieves detailed information about one incident.
    """

    print("\n[TOOL CALLED] get_incident")
    print(f"Ticket ID: {ticket_id}")

    incident_database = {

        "INC-6731": {
            "ticket_id": "INC-6731",
            "server": "APP-PROD-12",
            "summary": "Connection pool exhaustion",
            "root_cause":
                "Application connection pool reached its maximum configured size.",
            "resolution":
                "Restart application connection pool",
            "permanent_fix":
                "Review pool sizing and investigate connections not being released.",
            "severity": "SEV2"
        },

        "INC-6625": {
            "ticket_id": "INC-6625",
            "server": "APP-PROD-12",
            "summary": "High CPU utilization",
            "root_cause":
                "Runaway application worker consumed excessive CPU.",
            "resolution":
                "Restart affected worker process",
            "permanent_fix":
                "Patch worker-thread handling logic.",
            "severity": "SEV3"
        }
    }

    if ticket_id not in incident_database:
        return {
            "error": f"Incident '{ticket_id}' not found."
        }

    return incident_database[ticket_id]


# ------------------------------------------------------------
# TOOL 4: CREATE INCIDENT
# ------------------------------------------------------------

def create_incident(summary: str, severity: str):
    """
    Simulates creating a new incident.

    IMPORTANT:
    In a real environment this tool would normally require
    authorization / approval because it changes state.
    """

    print("\n[TOOL CALLED] create_incident")
    print(f"Summary : {summary}")
    print(f"Severity: {severity}")

    # Simulated ticket generation
    new_ticket = {
        "ticket_id": "INC-7001",
        "summary": summary,
        "severity": severity,
        "status": "OPEN"
    }

    return new_ticket


# ============================================================
# AGENT
# ============================================================

def investigate_server(server: str):

    print("\n" + "=" * 70)
    print("AGENT STARTED")
    print("=" * 70)

    print(
        f"\nUSER REQUEST:\n"
        f"{server} is responding slowly. Investigate."
    )

    # --------------------------------------------------------
    # STEP 1 — REASON
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("REASON")
    print("-" * 70)

    print(
        "The application is slow. "
        "I first need current infrastructure health data."
    )

    # --------------------------------------------------------
    # STEP 2 — TOOL CALL
    # --------------------------------------------------------

    health = get_server_health(server)

    # --------------------------------------------------------
    # STEP 3 — OBSERVATION
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("OBSERVATION")
    print("-" * 70)

    if "error" in health:
        print(health["error"])
        return

    print(f"CPU            : {health['cpu']}%")
    print(f"Memory         : {health['memory']}%")
    print(f"DB Connections : {health['db_connections']}%")
    print(f"Status         : {health['status']}")

    # --------------------------------------------------------
    # STEP 4 — UPDATE STATE
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("UPDATE STATE")
    print("-" * 70)

    suspected_issue = None

    if health["db_connections"] >= 95:

        suspected_issue = "database connection exhaustion"

        print(
            "Database connections are critically high."
        )

        print(
            "This may be causing application requests "
            "to wait for available database connections."
        )

    elif health["cpu"] >= 90:

        suspected_issue = "high CPU"

        print(
            "CPU utilization is critically high."
        )

    elif health["memory"] >= 90:

        suspected_issue = "memory pressure"

        print(
            "Memory utilization is critically high."
        )

    else:

        print(
            "No obvious infrastructure bottleneck was detected."
        )

    # --------------------------------------------------------
    # STEP 5 — NEXT DECISION
    # --------------------------------------------------------

    if suspected_issue:

        print("\n" + "-" * 70)
        print("NEXT DECISION")
        print("-" * 70)

        print(
            "Search historical incidents for similar symptoms."
        )

        query = (
            f"{server} database connections high"
            if suspected_issue == "database connection exhaustion"
            else f"{server} {suspected_issue}"
        )

        # ----------------------------------------------------
        # SECOND TOOL CALL
        # ----------------------------------------------------

        matches = search_incidents(query)

        # ----------------------------------------------------
        # SECOND OBSERVATION
        # ----------------------------------------------------

        print("\n" + "-" * 70)
        print("OBSERVATION")
        print("-" * 70)

        if not matches:

            print(
                "No similar historical incident found."
            )

            print(
                "\nRECOMMENDATION:"
                "\nEscalate for deeper investigation."
            )

            return

        print(
            f"Found {len(matches)} possible historical incident(s)."
        )

        best_match = matches[0]

        print(
            f"\nBest Match:"
            f"\nTicket     : {best_match['ticket_id']}"
            f"\nTitle      : {best_match['title']}"
            f"\nResolution : {best_match['resolution']}"
        )

        # ----------------------------------------------------
        # THIRD DECISION
        # ----------------------------------------------------

        print("\n" + "-" * 70)
        print("NEXT DECISION")
        print("-" * 70)

        print(
            "Retrieve the full incident record before "
            "making a recommendation."
        )

        incident = get_incident(
            best_match["ticket_id"]
        )

        # ----------------------------------------------------
        # THIRD OBSERVATION
        # ----------------------------------------------------

        print("\n" + "-" * 70)
        print("OBSERVATION")
        print("-" * 70)

        print(
            f"Incident      : {incident['ticket_id']}"
        )

        print(
            f"Root Cause    : {incident['root_cause']}"
        )

        print(
            f"Resolution    : {incident['resolution']}"
        )

        print(
            f"Permanent Fix : {incident['permanent_fix']}"
        )

        # ----------------------------------------------------
        # FINAL REASONING
        # ----------------------------------------------------

        print("\n" + "-" * 70)
        print("FINAL REASONING")
        print("-" * 70)

        print(
            "Current symptoms strongly resemble "
            "a previous connection-pool exhaustion incident."
        )

        # ----------------------------------------------------
        # FINAL RECOMMENDATION
        # ----------------------------------------------------

        print("\n" + "=" * 70)
        print("AGENT RECOMMENDATION")
        print("=" * 70)

        print(
            f"""
Likely issue:
Database connection pool exhaustion.

Evidence:
- CPU: {health['cpu']}%
- Memory: {health['memory']}%
- DB Connections: {health['db_connections']}%
- Server status: {health['status']}

Historical evidence:
- Incident: {incident['ticket_id']}
- Root cause: {incident['root_cause']}

Recommended immediate action:
{incident['resolution']}

Recommended follow-up:
{incident['permanent_fix']}
"""
        )


# ============================================================
# RUN THE DEMO
# ============================================================

investigate_server("APP-PROD-12")


AGENT STARTED

USER REQUEST:
APP-PROD-12 is responding slowly. Investigate.

----------------------------------------------------------------------
REASON
----------------------------------------------------------------------
The application is slow. I first need current infrastructure health data.

[TOOL CALLED] get_server_health
Server: APP-PROD-12

----------------------------------------------------------------------
OBSERVATION
----------------------------------------------------------------------
CPU            : 96%
Memory         : 71%
DB Connections : 100%
Status         : DEGRADED

----------------------------------------------------------------------
UPDATE STATE
----------------------------------------------------------------------
Database connections are critically high.
This may be causing application requests to wait for available database connections.

----------------------------------------------------------------------
NEXT DECISION
--------------------------------

USER:
"APP-PROD-12 is responding slowly. Investigate."

        ↓

AGENT REASONING:
"I need current server health."

        ↓

TOOL:
get_server_health("APP-PROD-12")

        ↓

OBSERVATION:
CPU = 96%
Memory = 71%
DB Connections = 100%
Status = DEGRADED

        ↓

UPDATED STATE:
"CPU is high, but DB connections are completely exhausted.
I should check whether similar incidents occurred before."

        ↓

TOOL:
search_incidents("APP-PROD-12 database connections high")

        ↓

OBSERVATION:
INC-6731
Connection pool exhaustion
Resolution: restart application connection pool

        ↓

NEXT DECISION:
Recommend restarting the application connection pool.